# Hyperparameter Tuning for PTM Prediction

Tunes hyperparameters for Random Forest and XGBoost using GridSearchCV or RandomizedSearchCV.

## Overview
- Uses 3-fold cross-validation
- Optimizes F1 score (better for imbalanced data)
- Can tune decision thresholds separately

## Configuration

In [ ]:
# File paths
TRAIN_FILE = "../data_engineered/onehot/train_with_features_onehot_split.csv"
VAL_FILE = "../data_engineered/onehot/val_with_features_onehot_split.csv"
import os
for sub in ["rf", "xgb", "svm"]:
    os.makedirs(os.path.join("..", "output", sub), exist_ok=True)

# Search method
SEARCH_METHOD = "random"  # "grid" or "random"
"""
Random Search: Randomly samples combinations from given distribution. More efficient.
Grid Search: Tests every combination of hyperparameters. Exhaustive. Takes longer.
"""

# Models to tune
TUNE_RF = True
TUNE_XGB = True
TUNE_SVM = True

# Cross-validation
CV_FOLDS = 6

# Random search iterations
N_ITER = 50

print("Configuration:")
print(f"  Search: {SEARCH_METHOD}")
print(f"  CV folds: {CV_FOLDS}")
print(f"  Random iterations: {N_ITER}")

In [ ]:
N_CORES = 16
# 1. Set Environment Variables (Must be done BEFORE importing sklearn/numpy)
os.environ["OMP_NUM_THREADS"] = str(N_CORES)
os.environ["MKL_NUM_THREADS"] = str(N_CORES)
os.environ["OPENBLAS_NUM_THREADS"] = str(N_CORES)
os.environ["VECLIB_MAXIMUM_THREADS"] = str(N_CORES)
os.environ["NUMEXPR_NUM_THREADS"] = str(N_CORES)

## Import Libraries

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.metrics import f1_score, make_scorer
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
import xgboost as xgb
import pickle
import warnings
warnings.filterwarnings("ignore")

## Load Data

In [ ]:
train_df = pd.read_csv(TRAIN_FILE)
val_df = pd.read_csv(VAL_FILE)
print(f"Train: {len(train_df):,}, Val: {len(val_df):,}")

## Prepare Data

In [ ]:
label_cols = ["S-glutathionylation", "S-nitrosylation", "S-palmitoylation"]
feature_cols = [col for col in train_df.columns if col not in ["ID", "Sequence"] + label_cols]

X_train_df = train_df[feature_cols].copy()
X_val_df = val_df[feature_cols].copy()

# Convert bool to int
for col in X_train_df.columns:
    if X_train_df[col].dtype == "bool":
        X_train_df[col] = X_train_df[col].astype(int)
        X_val_df[col] = X_val_df[col].astype(int)

# Remove non-numeric
non_numeric = X_train_df.select_dtypes(include=["object"]).columns
if len(non_numeric) > 0:
    print(f"⚠️ Removing {len(non_numeric)} non-numeric columns")
    X_train_df = X_train_df.drop(columns=non_numeric)
    X_val_df = X_val_df.drop(columns=non_numeric)

X_train = X_train_df.values.astype(float)
y_train = train_df[label_cols].values
X_val = X_val_df.values.astype(float)
y_val = val_df[label_cols].values

print(f"Features: {X_train.shape[1]}")

## Random Forest Tuning

In [ ]:
if TUNE_RF:
    print("\n" + "="*80)
    print("TUNING RANDOM FOREST")
    print("="*80)
    
    rf_models = {}
    rf_params = {}
    
    # Parameter grid
    if SEARCH_METHOD == "grid":
        param_grid = {
            "n_estimators": [200, 300, 400],
            "max_depth": [None, 10, 20],
            "min_samples_split": [2, 5, 10],
            "min_samples_leaf": [1, 2, 4],
            "max_features": ["sqrt", "log2"]
        }
    else:
        param_grid = {
            "n_estimators": [100, 200, 300, 400, 500],
            "max_depth": [None, 5, 10, 15, 20, 30],
            "min_samples_split": [2, 5, 10, 15],
            "min_samples_leaf": [1, 2, 4, 8],
            "max_features": ["sqrt", "log2", None]
        }
    
    for idx, label in enumerate(label_cols):
        print(f"\n[{label}]")
        
        # IMPORTANT: Set n_jobs=1 in base estimator to avoid nested parallelism
        # The outer GridSearchCV/RandomizedSearchCV will use N_CORES (16 cores)
        # Setting n_jobs in both places would cause 16 × 16 = 256 threads!
        rf_base = RandomForestClassifier(
            class_weight="balanced",
            random_state=42,
            n_jobs=1  # Changed from N_CORES to avoid nested parallelism
        )
        
        f1_scorer = make_scorer(f1_score)
        
        if SEARCH_METHOD == "grid":
            search = GridSearchCV(rf_base, param_grid, scoring=f1_scorer, cv=CV_FOLDS, n_jobs=N_CORES, verbose=3)
        else:
            search = RandomizedSearchCV(rf_base, param_grid, n_iter=N_ITER, scoring=f1_scorer, cv=CV_FOLDS, n_jobs=N_CORES, random_state=42, verbose=3)
        
        search.fit(X_train, y_train[:, idx])
        
        print(f"  Best params: {search.best_params_}")
        print(f"  Best CV F1: {search.best_score_:.4f}")
        
        rf_models[label] = search.best_estimator_
        rf_params[label] = search.best_params_
    
    # Save
    with open("../output/rf/rf_tuned_models.pkl", "wb") as f:
        pickle.dump(rf_models, f)
    print("\n✓ Saved rf_tuned_models.pkl")

  Best params: {'n_estimators': 400, 'min_samples_split': 10, 'min_samples_leaf': 8, 'max_features': 'log2', 'max_depth': 20}
  Best CV F1: 0.2031

[S-nitrosylation]
Fitting 6 folds for each of 50 candidates, totalling 300 fits
[CV 5/6] END max_depth=10, max_features=log2, min_samples_leaf=4, min_samples_split=5, n_estimators=100;, score=0.169 total time=   4.8s
[CV 4/6] END max_depth=None, max_features=sqrt, min_samples_leaf=8, min_samples_split=5, n_estimators=100;, score=0.105 total time=   9.2s
[CV 2/6] END max_depth=10, max_features=log2, min_samples_leaf=8, min_samples_split=5, n_estimators=400;, score=0.178 total time=  17.7s
[CV 6/6] END max_depth=20, max_features=sqrt, min_samples_leaf=1, min_samples_split=2, n_estimators=500;, score=0.176 total time=  43.4s
[CV 4/6] END max_depth=30, max_features=None, min_samples_leaf=8, min_samples_split=2, n_estimators=300;, score=0.089 total time= 3.2min
[CV 2/6] END max_depth=5, max_features=sqrt, min_samples_leaf=1, min_samples_split=2,

## XGBoost Tuning

In [ ]:
if TUNE_XGB:
    print("\n" + "="*80)
    print("TUNING XGBOOST")
    print("="*80)
    
    xgb_models = {}
    xgb_params = {}
    
    # Parameter grid
    if SEARCH_METHOD == "grid":
        param_grid = {
            "n_estimators": [200, 300, 400],
            "max_depth": [4, 6, 8],
            "learning_rate": [0.05, 0.1, 0.2],
            "subsample": [0.7, 0.8, 0.9],
            "colsample_bytree": [0.7, 0.8, 0.9]
        }
    else:
        param_grid = {
            "n_estimators": [100, 200, 300, 400, 500],
            "max_depth": [3, 4, 5, 6, 7, 8, 10],
            "learning_rate": [0.01, 0.05, 0.1, 0.15, 0.2, 0.3],
            "subsample": [0.6, 0.7, 0.8, 0.9, 1.0],
            "colsample_bytree": [0.6, 0.7, 0.8, 0.9, 1.0],
            "gamma": [0, 0.1, 0.2, 0.3],
            "reg_alpha": [0, 0.1, 0.5, 1],
            "reg_lambda": [0, 0.1, 0.5, 1]
        }
    
    for idx, label in enumerate(label_cols):
        print(f"\n[{label}]")
        
        # Calculate scale_pos_weight
        pos = y_train[:, idx].sum()
        neg = len(y_train) - pos
        scale_pos_weight = neg / pos
        
        # IMPORTANT: Set n_jobs=1 in base estimator to avoid nested parallelism
        # The outer GridSearchCV/RandomizedSearchCV will use N_CORES (16 cores)
        xgb_base = xgb.XGBClassifier(
            scale_pos_weight=scale_pos_weight,
            random_state=42,
            n_jobs=1,  # Changed from N_CORES to avoid nested parallelism
            eval_metric="logloss"
        )
        
        f1_scorer = make_scorer(f1_score)
        
        if SEARCH_METHOD == "grid":
            search = GridSearchCV(xgb_base, param_grid, scoring=f1_scorer, cv=CV_FOLDS, n_jobs=N_CORES, verbose=3)
        else:
            search = RandomizedSearchCV(xgb_base, param_grid, n_iter=N_ITER, scoring=f1_scorer, cv=CV_FOLDS, n_jobs=N_CORES, random_state=42, verbose=3)
        
        search.fit(X_train, y_train[:, idx])
        
        print(f"  Best params: {search.best_params_}")
        print(f"  Best CV F1: {search.best_score_:.4f}")
        
        xgb_models[label] = search.best_estimator_
        xgb_params[label] = search.best_params_
    
    # Save
    with open("../output/xgb/xgb_tuned_models.pkl", "wb") as f:
        pickle.dump(xgb_models, f)
    print("\n✓ Saved xgb_tuned_models.pkl")

## SVM Tuning

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)

In [ ]:
if TUNE_SVM:
    print("\n" + "="*80)
    print("TUNING SVM")
    print("="*80)

    svm_models = {}
    svm_params = {}

    # Parameter grid
    if SEARCH_METHOD == "grid":
        param_grid = {
            "C": [0.1, 1, 10],
            "kernel": ["linear", "rbf", "poly"],
            "gamma": ["scale", "auto"],
            "degree": [2, 3, 4]  # only used for poly kernel
        }
    else:
        param_grid = {
            "C": [0.01, 0.1, 1, 10, 100],
            "kernel": ["linear", "rbf", "poly", "sigmoid"],
            "gamma": ["scale", "auto"],
            "degree": [2, 3, 4, 5],
        }

    for idx, label in enumerate(label_cols):
        print(f"\n[{label}]")

        svm_base = SVC(
            class_weight="balanced",
            probability=False,    # set True only if needed for calibration
            random_state=42
        )

        f1_scorer = make_scorer(f1_score)

        # Choose search method
        if SEARCH_METHOD == "grid":
            search = GridSearchCV(
                svm_base,
                param_grid,
                scoring=f1_scorer,
                cv=CV_FOLDS,
                n_jobs=N_CORES,
                verbose=3
            )
        else:
            search = RandomizedSearchCV(
                svm_base,
                param_grid,
                n_iter=N_ITER,
                scoring=f1_scorer,
                cv=CV_FOLDS,
                n_jobs=N_CORES,
                random_state=42,
                verbose=3
            )

        # Fit
        search.fit(X_train, y_train[:, idx])

        print(f"  Best params: {search.best_params_}")
        print(f"  Best CV F1: {search.best_score_:.4f}")

        # Store best model + params
        svm_models[label] = search.best_estimator_
        svm_params[label] = search.best_params_

    # Save results
    with open("../output/svm/svm_tuned_models.pkl", "wb") as f:
        pickle.dump(svm_models, f)

    print("\n✓ Saved svm_tuned_models.pkl")


## Summary

In [ ]:
print("\n" + "="*80)
print("HYPERPARAMETER TUNING COMPLETE")
print("="*80)
print("\nTuned models saved. Use these instead of default models for better performance.")